# Phase 5: predicted MuSiQue A/B/C/D pilot
Uses global dev SQLite FTS5/BM25 without reranker, frozen phase-4 raw DeBERTa gate, SRO relation gate, and Kaggle `openai/gpt-5.4-mini-2026-03-17` for QA/judge. This is a **variant**, not the paper reproduction. Prior-hop references use each arm's predicted answers, substituted into gold-derived templates. A 30-QA pilot with a 600-call guard.
Runner SHA256: `43D9F22E02B96CD04227D3183684B0D9042F5D666F562358E1481C5A4024A670`.


In [1]:
# Kaggle Benchmarks starts with a minimal Python environment.
import importlib.util, subprocess, sys
if importlib.util.find_spec('torch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--index-url', 'https://download.pytorch.org/whl/cpu', 'torch==2.15.0+cpu'])
if importlib.util.find_spec('transformers') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'transformers==5.24.1', 'sentencepiece==0.2.3'])


Looking in indexes: https://download.pytorch.org/whl/cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/209.5 MB ? eta -:--:--

   ━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/209.5 MB 164.6 MB/s eta 0:00:02

   ━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.0/209.5 MB 162.5 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 103.0/209.5 MB 182.5 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━ 153.1/209.5 MB 198.8 MB/s eta 0:00:01

... (170 outputs left out)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━ 10/11 [transformers]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━ 10/11 [transformers]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━ 10/11 [transformers]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [transformers]


In [2]:
"""Kaggle phase-5 oracle-normalized A/B/C/D variant.

This is deliberately not the paper reproduction: it uses a global MuSiQue-dev
SQLite FTS5 corpus without a reranker and Kaggle GPT-5.4-mini for QA/judging.
The mapped sub-questions can contain gold prior-hop answers. Never label this
run non-oracle. Each completed QA is written with all four arms together.
"""

import hashlib
import copy
import json
import os
import re
import sqlite3
import statistics
import time
import unicodedata
from collections import Counter
from pathlib import Path

import kaggle_benchmarks as kbench
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer


MODEL_ID = "openai/gpt-5.4-mini-2026-03-17"
DATA = Path("/kaggle/input/datasets/chnglng/fact-grounding-sro-phase-5-musique-dev-20261125")
WEIGHTS = Path("/kaggle/input/datasets/chnglng/fg-sro-phase-4c-checkpoints-20261125")
RAW_CHECKPOINT = WEIGHTS / "phase4c_raw/checkpoints/checkpoint-2639"
TOKENIZER_CHECKPOINT = WEIGHTS / "phase4c_sro/checkpoints/checkpoint-13195"
OUTPUT = Path("/kaggle/working/phase5_predicted_v2")
PRIOR_OUTPUT = None  # A later notebook version may mount a completed prior version here.
QA_HASH = "15FA63794D18A94CE12411ACA6E2327E65B6E83B0B1490EFAB3F1962E48ABF3B"
MAPPING_HASH = "F497D42592CEC36474D99A6D51563F656D2C8019A086C4CD9D8537EB40D59965"
SEED = "phase5_bm25_oracle_v1"
INITIAL_K = 10
INITIAL_KEEP = 3
EXTRA_K = 20
EXTRA_KEEP = 3
MAX_CHARS = 4000
MAX_CALLS = 600
QA_LIMIT = 30
QA_START = 0
QA_STOP = None
NORMALIZATION_MODE = "predicted_substitution"
TOKEN = re.compile(r"[^\W_]+", re.UNICODE)
REF = re.compile(r"#(\d+)")


class BudgetExhausted(Exception):
    pass


def sha_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest().upper()


def load_jsonl(path, expected_hash):
    if sha_file(path) != expected_hash:
        raise ValueError(f"hash mismatch: {path}")
    with path.open(encoding="utf-8") as source:
        return [json.loads(line) for line in source]


def sample_qids(qa_rows, limit):
    groups = {}
    for row in qa_rows:
        groups.setdefault(len(row["question_decomposition"]), []).append(row["id"])
    if limit is None:
        return sorted(row["id"] for row in qa_rows)
    if limit % len(groups):
        raise ValueError("pilot limit must divide evenly by hop-count strata")
    per_group = limit // len(groups)
    chosen = []
    for _, qids in sorted(groups.items()):
        chosen += sorted(qids, key=lambda qid: hashlib.sha256(f"{SEED}:{qid}".encode()).digest())[:per_group]
    return sorted(chosen)


def sro_query(mapped):
    if mapped["sro_status"] != "ok":
        return mapped["oracle_normalized_question"]
    fields = []
    for triple in mapped["sro"]["triples"]:
        fields += [str(triple[key]) for key in ("subject", "relation", "object", "condition")
                   if triple.get(key) and triple[key] != "?"]
    return " ".join(fields)


def fts_query(value):
    return " OR ".join('"' + term + '"' for term in TOKEN.findall(value.casefold()))


class Retriever:
    def __init__(self, qa_rows):
        self.db = sqlite3.connect(":memory:")
        self.db.execute("CREATE VIRTUAL TABLE docs USING fts5(title, body, tokenize='unicode61')")
        self.by_key = {}
        for qa in qa_rows:
            for passage in qa["paragraphs"]:
                key = (passage["title"], passage["paragraph_text"])
                if key not in self.by_key:
                    self.by_key[key] = len(self.by_key) + 1
        self.db.executemany("INSERT INTO docs(rowid,title,body) VALUES(?,?,?)",
                            ((doc_id, *key) for key, doc_id in self.by_key.items()))
        self.db.commit()
        self.by_id = {doc_id: key for key, doc_id in self.by_key.items()}

    def search(self, query, limit):
        match = fts_query(query)
        if not match:
            return []
        rows = self.db.execute("SELECT rowid,title,body FROM docs WHERE docs MATCH ? "
                               "ORDER BY bm25(docs) LIMIT ?", (match, limit))
        return [{"id": row_id, "title": title, "text": body} for row_id, title, body in rows]

    def support_id(self, qa, hop_index):
        idx = qa["question_decomposition"][hop_index]["paragraph_support_idx"]
        passage = qa["paragraphs"][idx]
        return self.by_key[(passage["title"], passage["paragraph_text"])]


def dedupe(passages):
    seen, result = set(), []
    for passage in passages:
        if passage["id"] not in seen:
            seen.add(passage["id"])
            result.append(passage)
    return result


def format_passages(passages):
    result, used = [], 0
    for passage in passages:
        entry = f"[{passage['id']}] {passage['title']}: {passage['text']}"
        if used + len(entry) > MAX_CHARS:
            break
        result.append(entry)
        used += len(entry)
    return "\n".join(result)


def clean_answer(value):
    return str(value).strip().strip("*_\"'`#->.,;: ")


def answer_metrics(gold, prediction, aliases=()):
    def tokens(value):
        value = unicodedata.normalize("NFKC", value).casefold()
        value = re.sub(r"[^\w\s]", " ", value, flags=re.UNICODE)
        return [token for token in value.split() if token not in {"a", "an", "the"}]

    predicted = tokens(prediction)
    candidates = [gold] + list(aliases)
    scores = []
    for candidate in candidates:
        expected = tokens(candidate)
        exact = int(predicted == expected)
        overlap = sum((Counter(predicted) & Counter(expected)).values())
        f1 = (2 * overlap / (len(predicted) + len(expected))
              if predicted or expected else 1.0)
        scores.append((exact, f1))
    return {"em": max(score[0] for score in scores),
            "token_f1": max(score[1] for score in scores)}


def replace_answer(text, gold, predicted):
    pattern = re.compile(r"(?<!\w)" + re.escape(gold) + r"(?!\w)", re.IGNORECASE)
    changed, count = pattern.subn(lambda _: predicted, text)
    if count:
        return changed, count
    # The source sometimes stores a terminal period in the answer while the
    # normalized question uses a question mark after the entity.
    trimmed = gold.rstrip(" .?!")
    if trimmed != gold and trimmed:
        pattern = re.compile(r"(?<!\w)" + re.escape(trimmed) + r"(?!\w)", re.IGNORECASE)
        return pattern.subn(lambda _: predicted, text)
    return changed, count


def render_hop(mapped, qa, predicted_answers):
    """Resolve #N from this arm's predictions; gold is used only to find template slots."""
    if NORMALIZATION_MODE == "oracle":
        return mapped
    if NORMALIZATION_MODE != "predicted_substitution":
        raise ValueError("invalid normalization mode")
    rendered = copy.deepcopy(mapped)
    refs = sorted({int(value) for value in REF.findall(mapped["decomposition_question"])})
    if any(ref < 1 or ref > len(predicted_answers) for ref in refs):
        raise ValueError("missing predicted prior-hop answer")
    for ref in refs:
        gold = qa["question_decomposition"][ref - 1]["answer"]
        predicted = predicted_answers[ref - 1] or "unknown"
        question, count = replace_answer(rendered["oracle_normalized_question"], gold, predicted)
        if count < 1:
            raise ValueError("gold-derived question template has no reference slot")
        rendered["oracle_normalized_question"] = question
        if rendered["sro_status"] == "ok":
            for triple in rendered["sro"]["triples"]:
                for field in ("subject", "relation", "object", "condition"):
                    if isinstance(triple.get(field), str):
                        triple[field], _ = replace_answer(triple[field], gold, predicted)
    return rendered


class LLM:
    def __init__(self, max_calls, model=None):
        self.model = model if model is not None else kbench.llms[MODEL_ID]
        self.max_calls = max_calls
        self.calls = 0
        self.cache = {}
        self.latencies = []
        self.usage = Counter()
        self.usage_unavailable = 0
        self.empty_responses = 0

    def prompt(self, kind, prompt):
        key = hashlib.sha256((MODEL_ID + "\n" + prompt).encode()).hexdigest()
        if key in self.cache:
            return self.cache[key]
        for _ in range(2):
            if self.calls >= self.max_calls:
                raise BudgetExhausted("frozen per-version call budget reached")
            started = time.monotonic()
            # Kaggle Benchmarks appends prompt() to the current chat. A fresh
            # chat prevents a prior judge's gold answer entering QA.
            with kbench.chats.new(f"phase5_{self.calls + 1}_{key[:12]}") as chat:
                result = str(self.model.prompt(prompt))
                usage = getattr(chat, "usage", None)
                fields = ("input_tokens", "output_tokens", "input_tokens_cost_nanodollars",
                          "output_tokens_cost_nanodollars", "total_backend_latency_ms")
                if usage is None or any(getattr(usage, field, None) is None for field in fields):
                    self.usage_unavailable += 1
                else:
                    self.usage.update({field: getattr(usage, field) for field in fields})
            self.latencies.append(time.monotonic() - started)
            self.calls += 1
            if result.strip():
                self.cache[key] = result
                return result
            self.empty_responses += 1
        raise ValueError(f"empty {kind} response after two isolated attempts")


class RawGate:
    def __init__(self):
        if not (RAW_CHECKPOINT / "model.safetensors").is_file():
            raise FileNotFoundError(RAW_CHECKPOINT / "model.safetensors")
        if not (TOKENIZER_CHECKPOINT / "tokenizer.json").is_file():
            raise FileNotFoundError(TOKENIZER_CHECKPOINT / "tokenizer.json")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_CHECKPOINT, local_files_only=True)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            RAW_CHECKPOINT, local_files_only=True).to(self.device).eval()
        if self.model.config.label2id.get("ANSWERABLE") != 1:
            raise ValueError("raw checkpoint label mapping mismatch")

    def check(self, question, passages):
        content = f"Sub-question: {question}\n\nPassages:\n{format_passages(passages)}"
        batch = self.tokenizer(content, max_length=512, truncation=True, return_tensors="pt").to(self.device)
        with torch.inference_mode():
            probability = torch.softmax(self.model(**batch).logits.float(), dim=-1)[0, 1].item()
        return {"status": "supported" if probability >= 0.5 else "unknown",
                "answerable_probability": probability, "evidence": []}


def parse_json_response(text):
    trimmed = text.strip()
    if trimmed.startswith("```"):
        trimmed = re.sub(r"^```(?:json)?\s*|\s*```$", "", trimmed)
    return json.loads(trimmed)


def sro_gate(llm, sro, passages):
    triples = sro["triples"]
    prompt = (
        "Judge whether EACH subject-relation-object triple is supported by the given passages. "
        "Use only these passages; do not use world knowledge. Return compact JSON with key 'triples', "
        "an array of objects with keys id,status,passage_id,span. status must be exactly "
        "supported, contradicted, or unknown. For supported/contradicted, span must be an exact "
        "substring of the cited passage text; for unknown set passage_id and span to null. "
        "Treat '?' as an unknown value to be found, not as evidence that the relation is unknown.\n"
        f"Triples: {json.dumps(triples, ensure_ascii=False)}\nPassages:\n{format_passages(passages)}"
    )
    result = parse_json_response(llm.prompt("sro_gate", prompt))
    statuses = []
    by_id = {passage["id"]: passage for passage in passages}
    outputs = result.get("triples")
    if not isinstance(outputs, list) or len(outputs) != len(triples):
        raise ValueError("SRO gate triple count mismatch")
    expected = [triple["id"] for triple in triples]
    if [item.get("id") for item in outputs] != expected:
        raise ValueError("SRO gate triple ID mismatch")
    for item in outputs:
        status = item.get("status")
        if status not in {"supported", "contradicted", "unknown"}:
            raise ValueError("invalid SRO gate status")
        if status == "unknown":
            if item.get("passage_id") is not None or item.get("span") is not None:
                raise ValueError("unknown relation cited evidence")
        else:
            passage_id = item.get("passage_id")
            if isinstance(passage_id, str) and passage_id.isdigit():
                passage_id = int(passage_id)
                item["passage_id"] = passage_id
            passage = by_id.get(passage_id)
            if passage is None or not item.get("span") or item["span"] not in passage["text"]:
                raise ValueError("unverified evidence span")
        statuses.append(item)
    overall = "unknown" if any(x["status"] == "unknown" for x in statuses) else (
        "contradicted" if any(x["status"] == "contradicted" for x in statuses) else "supported")
    return {"status": overall, "evidence": statuses}


def check_gate(arm, raw_gate, llm, mapped, passages):
    if arm in {"A", "C"} or mapped["sro_status"] != "ok":
        result = raw_gate.check(mapped["oracle_normalized_question"], passages)
        result["fallback"] = arm in {"B", "D"}
        return result
    try:
        return sro_gate(llm, mapped["sro"], passages)
    except BudgetExhausted:
        raise
    except (ValueError, KeyError, TypeError) as exc:
        result = raw_gate.check(mapped["oracle_normalized_question"], passages)
        result["fallback"] = "sro_gate_error"
        result["error"] = f"{type(exc).__name__}: {exc}"
        return result


def extra_query(arm, original_question, mapped, first_gate):
    if arm in {"B", "D"} and mapped["sro_status"] == "ok":
        unknown = {row["id"] for row in first_gate["evidence"] if row["status"] == "unknown"}
        parts = []
        for triple in mapped["sro"]["triples"]:
            if triple["id"] in unknown:
                parts += [str(triple[key]) for key in ("subject", "relation", "object", "condition")
                          if triple.get(key) and triple[key] != "?"]
        return " ".join(parts) + " " + original_question
    return mapped["oracle_normalized_question"] + " " + original_question


def answer_hop(llm, question, passages):
    prompt = ("Answer ONLY this specific question using the passages below. "
              "Give a SHORT answer (just the entity or fact, no explanation).\n\n"
              f"Passages:\n{format_passages(passages)}\n\nQuestion: {question}\n\nAnswer:")
    return clean_answer(llm.prompt("hop_answer", prompt))


def final_answer(llm, question, subquestions, answers):
    pairs = "\n".join(f"Q{i + 1}: {q}\nA{i + 1}: {a}" for i, (q, a) in enumerate(zip(subquestions, answers)))
    prompt = ("Given the following intermediate answers to sub-questions, answer the original question.\n\n"
              f"{pairs}\n\nOriginal question: {question}\n"
              "Give a SHORT answer (just the entity or fact, no explanation).\nFinal answer:")
    return clean_answer(llm.prompt("final_answer", prompt))


def judge(llm, question, gold, prediction):
    prompt = ("You are evaluating a question-answering system.\n\n"
              f"Question: {question}\nGold answer: {gold}\nPredicted answer: {prediction}\n\n"
              "Is the predicted answer semantically equivalent to the gold answer? Accept paraphrases, "
              "aliases, and partial matches that capture the key entity/fact. Reject answers that name "
              "a different entity or miss the key fact. Reply with ONLY YES or NO.")
    response = clean_answer(llm.prompt("judge", prompt)).upper()
    if response not in {"YES", "NO"}:
        raise ValueError(f"invalid judge response: {response[:80]}")
    return response == "YES"


def run_arm(arm, qa, mapped_hops, retriever, raw_gate, llm):
    passages, hop_results, answers, questions = [], [], [], []
    for hop_index, mapped in enumerate(mapped_hops):
        mapped = render_hop(mapped, qa, answers)
        raw_question = mapped["oracle_normalized_question"]
        query = raw_question if arm in {"A", "B"} else sro_query(mapped)
        initial = retriever.search(query, INITIAL_K)
        passages = dedupe(passages + initial[:INITIAL_KEEP])
        before = check_gate(arm, raw_gate, llm, mapped, passages)
        extra_ids = []
        after = before
        if before["status"] == "unknown":
            candidates = retriever.search(extra_query(arm, qa["question"], mapped, before), EXTRA_K)
            existing = {passage["id"] for passage in passages}
            extra = [passage for passage in candidates if passage["id"] not in existing][:EXTRA_KEEP]
            extra_ids = [passage["id"] for passage in extra]
            passages = dedupe(passages + extra)
            after = check_gate(arm, raw_gate, llm, mapped, passages)
        answer = answer_hop(llm, raw_question, passages)
        answers.append(answer)
        questions.append(raw_question)
        support_id = retriever.support_id(qa, hop_index)
        hop_results.append({
            "row_index": mapped["row_index"], "query": query,
            "initial_doc_ids": [item["id"] for item in initial], "extra_doc_ids": extra_ids,
            "retained_doc_ids": [item["id"] for item in passages],
            "support_doc_id": support_id, "initial_recall": support_id in {x["id"] for x in initial},
            "final_evidence_recall": support_id in {x["id"] for x in passages},
            "gate_before": before, "gate_after": after, "answer": answer,
        })
    prediction = final_answer(llm, qa["question"], questions, answers)
    return {"hops": hop_results, "prediction": prediction,
            "correct": judge(llm, qa["question"], qa["answer"], prediction),
            **answer_metrics(qa["answer"], prediction, qa.get("answer_aliases", []))}


def audit_records(records, qa_by_id, mapping_by_id, retriever):
    seen = set()
    for record in records:
        qid = record["qid"]
        if qid in seen or qid not in qa_by_id or set(record["arms"]) != set("ABCD"):
            raise ValueError("duplicate/unknown/unpaired QA record")
        seen.add(qid)
        qa = qa_by_id[qid]
        if record["gold_answer"] != qa["answer"]:
            raise ValueError("gold answer mismatch")
        for arm in "ABCD":
            hops = record["arms"][arm]["hops"]
            if len(hops) != len(mapping_by_id[qid]):
                raise ValueError("hop count mismatch")
            for index, hop in enumerate(hops):
                support_id = retriever.support_id(qa, index)
                if hop["row_index"] != mapping_by_id[qid][index]["row_index"] or hop["support_doc_id"] != support_id:
                    raise ValueError("mapping/support ID mismatch")
                if hop["initial_recall"] != (support_id in hop["initial_doc_ids"]):
                    raise ValueError("initial recall mismatch")
                if hop["final_evidence_recall"] != (support_id in hop["retained_doc_ids"]):
                    raise ValueError("final evidence recall mismatch")
                for gate in (hop["gate_before"], hop["gate_after"]):
                    for item in gate.get("evidence", []):
                        if item["status"] in {"supported", "contradicted"}:
                            passage_id = item["passage_id"]
                            if passage_id not in hop["retained_doc_ids"] or passage_id not in retriever.by_id:
                                raise ValueError("evidence passage not retained")
                            if item["span"] not in retriever.by_id[passage_id][1]:
                                raise ValueError("evidence span mismatch")
    return {"status": "passed", "qa_records": len(records),
            "hop_records_per_arm": sum(len(mapping_by_id[qid]) for qid in seen)}


def summarize(records, retriever, llm, target_count):
    report = {"status": "complete" if len(records) == target_count else "partial",
              "model_id": MODEL_ID, "oracle_normalization": NORMALIZATION_MODE == "oracle",
              "normalization_mode": NORMALIZATION_MODE,
              "template_origin": "gold_normalized_csv_then_predicted_answer_substitution"
              if NORMALIZATION_MODE == "predicted_substitution" else "gold_normalized_csv",
              "variant": "global_dev_corpus_sqlite_fts5_bm25_no_reranker",
              "target_qa": target_count, "completed_qa": len(records), "global_passages": len(retriever.by_key),
              "model_calls": llm.calls, "cache_size": len(llm.cache),
              "empty_model_responses": llm.empty_responses,
              "latency_p50_seconds": statistics.median(llm.latencies) if llm.latencies else None,
              "latency_p95_seconds": sorted(llm.latencies)[int(.95 * (len(llm.latencies)-1))] if llm.latencies else None,
              "usage": dict(llm.usage) if not llm.usage_unavailable else None,
              "usage_unavailable_calls": llm.usage_unavailable}
    for arm in "ABCD":
        arm_rows = [record["arms"][arm] for record in records]
        hops = [hop for row in arm_rows for hop in row["hops"]]
        report[arm] = {"qa_accuracy": sum(row["correct"] for row in arm_rows) / len(arm_rows) if arm_rows else None,
                       "correct_qa": sum(row["correct"] for row in arm_rows),
                       "em": sum(row.get("em", 0) for row in arm_rows) / len(arm_rows) if arm_rows else None,
                       "token_f1": sum(row.get("token_f1", 0) for row in arm_rows) / len(arm_rows) if arm_rows else None,
                       "hops": len(hops),
                       "initial_recall": sum(hop["initial_recall"] for hop in hops) / len(hops) if hops else None,
                       "final_evidence_recall": sum(hop["final_evidence_recall"] for hop in hops) / len(hops) if hops else None,
                       "extra_calls": sum(bool(hop["extra_doc_ids"]) for hop in hops),
                       "retrieval_calls": len(hops) + sum(bool(hop["extra_doc_ids"]) for hop in hops),
                       "new_support_after_extra": sum(
                           not hop["initial_recall"] and hop["final_evidence_recall"] for hop in hops),
                       "gate_fallbacks": sum(bool(hop["gate_before"].get("fallback")) for hop in hops),
                       "contradicted_before": sum(hop["gate_before"]["status"] == "contradicted" for hop in hops),
                       "answerability_flag_rate": sum(hop["gate_before"]["status"] == "unknown" for hop in hops) / len(hops) if hops else None,
                       "unknown_before": sum(hop["gate_before"]["status"] == "unknown" for hop in hops)}
    report["paired"] = {}
    for arm in "BCD":
        improved = sum(row["arms"][arm]["correct"] and not row["arms"]["A"]["correct"]
                       for row in records)
        worsened = sum(row["arms"]["A"]["correct"] and not row["arms"][arm]["correct"]
                       for row in records)
        report["paired"][f"{arm}_minus_A"] = {
            "improved": improved, "worsened": worsened,
            "net_correct": improved - worsened,
            "accuracy_delta": (improved - worsened) / len(records) if records else None,
        }
    return report


def main(model=None):
    qa_rows = load_jsonl(DATA / "musique_ans_v1.0_dev.jsonl", QA_HASH)
    mapping = load_jsonl(DATA / "oracle_mapping.jsonl", MAPPING_HASH)
    qa_by_id = {row["id"]: row for row in qa_rows}
    mapping_by_id = {}
    for row in mapping:
        mapping_by_id.setdefault(row["qid"], []).append(row)
    if set(qa_by_id) != set(mapping_by_id):
        raise ValueError("QA/mapping ID mismatch")
    qids = sample_qids(qa_rows, QA_LIMIT)[QA_START:QA_STOP]
    if not qids:
        raise ValueError("empty QA slice")
    retriever = Retriever(qa_rows)
    raw_gate = RawGate()
    llm = LLM(MAX_CALLS, model)
    completed = []
    prior_total_calls = 0
    if PRIOR_OUTPUT is not None:
        prior = Path(PRIOR_OUTPUT)
        prior_manifest = json.loads((prior / "manifest.json").read_text(encoding="utf-8"))
        if (prior_manifest["model_id"] != MODEL_ID or
                prior_manifest["qa_sha256"] != QA_HASH or
                prior_manifest["mapping_sha256"] != MAPPING_HASH or
                prior_manifest.get("normalization_mode", "oracle") != NORMALIZATION_MODE):
            raise ValueError("prior phase-5 protocol mismatch")
        if sha_file(prior / "paired.jsonl") != prior_manifest["paired_sha256"]:
            raise ValueError("prior paired artifact hash mismatch")
        with (prior / "paired.jsonl").open(encoding="utf-8") as source:
            completed = [json.loads(line) for line in source]
        prior_total_calls = prior_manifest.get("cumulative_model_calls", prior_manifest["model_calls"])
        seen = [row["qid"] for row in completed]
        if (len(seen) != len(set(seen)) or not set(seen) <= set(qids) or
                any(set(row["arms"]) != set("ABCD") for row in completed)):
            raise ValueError("prior rows are duplicate, outside target or unpaired")
        for row in completed:
            qa = qa_by_id[row["qid"]]
            row["answer_aliases"] = qa.get("answer_aliases", [])
            for arm in "ABCD":
                row["arms"][arm].update(answer_metrics(
                    qa["answer"], row["arms"][arm]["prediction"], row["answer_aliases"]))
    OUTPUT.mkdir(parents=True, exist_ok=False)
    path = OUTPUT / "paired.jsonl"
    completed_ids = {row["qid"] for row in completed}
    stop_reason = None
    with path.open("x", encoding="utf-8", newline="\n") as target:
        for row in completed:
            target.write(json.dumps(row, ensure_ascii=False) + "\n")
        target.flush()
        os.fsync(target.fileno())
        for qid in qids:
            if qid in completed_ids:
                continue
            qa = qa_by_id[qid]
            before_calls = llm.calls
            before_time = time.monotonic()
            try:
                arms = {arm: run_arm(arm, qa, mapping_by_id[qid], retriever, raw_gate, llm)
                        for arm in "ABCD"}
            except BudgetExhausted as exc:
                stop_reason = str(exc)
                break
            except Exception as exc:
                stop_reason = f"qa_failed:{qid}:{type(exc).__name__}:{exc}"
                break
            row = {"qid": qid, "n_hops": len(mapping_by_id[qid]), "gold_answer": qa["answer"],
                   "answer_aliases": qa.get("answer_aliases", []),
                   "arms": arms, "new_model_calls": llm.calls - before_calls,
                   "elapsed_seconds": time.monotonic() - before_time}
            target.write(json.dumps(row, ensure_ascii=False) + "\n")
            target.flush()
            os.fsync(target.fileno())
            completed.append(row)
            print(json.dumps({"completed": len(completed), "target": len(qids),
                              "calls": llm.calls, "qid": qid}), flush=True)
    audit = audit_records(completed, qa_by_id, mapping_by_id, retriever)
    report = summarize(completed, retriever, llm, len(qids))
    report["audit"] = audit
    report.update({"stop_reason": stop_reason, "qa_sha256": QA_HASH,
                   "mapping_sha256": MAPPING_HASH, "raw_checkpoint": str(RAW_CHECKPOINT),
                   "tokenizer_checkpoint": str(TOKENIZER_CHECKPOINT),
                   "prior_output": str(PRIOR_OUTPUT) if PRIOR_OUTPUT else None,
                   "carried_qa": len(completed_ids),
                   "cumulative_model_calls": prior_total_calls + llm.calls,
                   "paired_sha256": sha_file(path), "configuration": {
                       "initial_k": INITIAL_K, "initial_keep": INITIAL_KEEP,
                       "extra_k": EXTRA_K, "extra_keep": EXTRA_KEEP,
                       "max_chars": MAX_CHARS, "max_calls": MAX_CALLS, "qa_limit": QA_LIMIT,
                       "qa_start": QA_START, "qa_stop": QA_STOP}})
    (OUTPUT / "manifest.json").write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print(json.dumps(report, ensure_ascii=False, indent=2), flush=True)
    return report


@kbench.task(name="phase5_predicted_pilot_20261125_13")
def run_task(llm) -> float:
    report = main(model=llm)
    return float(report["D"]["qa_accuracy"] or 0.0)

if __name__ == "__main__":
    run_task.run(kbench.llms[MODEL_ID])


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

{"completed": 1, "target": 30, "calls": 10, "qid": "2hop__17827_54024"}


{"completed": 2, "target": 30, "calls": 25, "qid": "2hop__25797_25855"}


{"completed": 3, "target": 30, "calls": 31, "qid": "2hop__450608_373421"}


{"completed": 4, "target": 30, "calls": 42, "qid": "2hop__511176_22458"}


{"completed": 5, "target": 30, "calls": 50, "qid": "2hop__5658_161105"}


... (22 outputs left out)

{"completed": 28, "target": 30, "calls": 455, "qid": "4hop2__9988_158279_70784_79935"}


{"completed": 29, "target": 30, "calls": 479, "qid": "4hop3__160230_470287_345497_72134"}


{"completed": 30, "target": 30, "calls": 511, "qid": "4hop3__312119_132409_146285_35031"}


{
  "status": "complete",
  "model_id": "openai/gpt-5.4-mini-2026-03-17",
  "oracle_normalization": false,
  "normalization_mode": "predicted_substitution",
  "template_origin": "gold_normalized_csv_then_predicted_answer_substitution",
  "variant": "global_dev_corpus_sqlite_fts5_bm25_no_reranker",
  "target_qa": 30,
  "completed_qa": 30,
  "global_passages": 21100,
  "model_calls": 511,
  "cache_size": 511,
  "empty_model_responses": 0,
  "latency_p50_seconds": 0.4908850110000458,
  "latency_p95_seconds": 0.840752273000362,
  "usage": {
    "input_tokens": 316150,
    "output_tokens": 7908,
    "input_tokens_cost_nanodollars": 235038900,
    "output_tokens_cost_nanodollars": 35586000,
    "total_backend_latency_ms": 253790
  },
  "usage_unavailable_calls": 0,
  "A": {
    "qa_accuracy": 0.4,
    "correct_qa": 12,
    "em": 0.26666666666666666,
    "token_f1": 0.40555555555555556,
    "hops": 90,
    "initial_recall": 0.7444444444444445,
    "final_evidence_recall": 0.7333333333333333,
